# `json2vec` Hello World

This notebook trains the smallest useful JSON2Vec model: two numeric Iris measurements predict the Iris species. The point is not accuracy. The point is to show the complete loop from records, to schema, to training, to prediction and embeddings.

Start with the normal training dependencies plus the bundled Iris dataset. The examples remove notebook logging noise so the rendered docs stay focused on model behavior.

In [1]:
import contextlib
import io
import logging

import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint
from sklearn.datasets import load_iris

import json2vec as j2v

logger.remove()

Build a tiny Polars frame from a balanced slice of Iris rows. `ROOT = "[*][*]"` is the JMESPath prefix used by batched data modules: the first `[*]` iterates batch items, and the second iterates records inside each item.

In [2]:
ROOT = "[*][*]"
iris = load_iris()
indices = [*range(0, 12), *range(50, 62), *range(100, 112)]

records = pl.DataFrame(
    {
        "sepal_length": iris.data[indices, 0].tolist(),
        "petal_length": iris.data[indices, 2].tolist(),
        "species": [iris.target_names[index] for index in iris.target[indices]],
    }
)

records.head()

sepal_length,petal_length,species
f64,f64,str
5.1,1.4,"""setosa"""
4.9,1.4,"""setosa"""
4.7,1.3,"""setosa"""
4.6,1.5,"""setosa"""
5.0,1.4,"""setosa"""


The schema declares exactly what the model should read. `Number` fields become numeric tensorfields, and the `Category` field is a supervised target because `target=True` hides it from the input and asks the model to decode it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("sepal_length", query=f"{ROOT}.sepal_length"),
    j2v.Number("petal_length", query=f"{ROOT}.petal_length"),
    j2v.Category("species", query=f"{ROOT}.species", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule.from_model(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)

Train for one deliberately small epoch. The tutorials keep batch and epoch counts hardcoded so the example remains quick to run in documentation builds.

In [4]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    logging.disable(logging.CRITICAL)
    try:
        lit.seed_everything(7, workers=True, verbose=False)
        trainer = lit.Trainer(
            accelerator="cpu",
            max_epochs=1,
            logger=False,
            enable_progress_bar=False,
            enable_model_summary=False,
            enable_checkpointing=False,
            limit_train_batches=1,
            limit_val_batches=1,
        )
        trainer.fit(model=model, datamodule=datamodule)
    finally:
        logging.disable(logging.NOTSET)

Prediction uses the same nested batch shape as training: each outer item is one observation, and each observation contains one record.

In [5]:
batch = [[record] for record in records.to_dicts()[:3]]

In [6]:
pprint(model.predict(batch))

{
│   'record/species': {
│   │   'state': {
│   │   │   'valued': [0.05262981355190277, 0.05262981355190277, 0.05262981355190277],
│   │   │   'null': [0.20542284846305847, 0.20542284846305847, 0.20542284846305847],
│   │   │   'padded': [0.5692998170852661, 0.5692998170852661, 0.5692998766899109],
│   │   │   'masked': [0.0808497741818428, 0.0808497741818428, 0.0808497741818428],
│   │   │   'other': [0.09179771691560745, 0.09179771691560745, 0.09179771691560745]
│   │   },
│   │   'content': {'value': [None, None, None], 'probability': [0.0, 0.0, 0.0], 'topk': [[], [], []]}
│   }
}

Embeddings are opt-in. Mutating the root `record` node with `embed=True` exposes a vector for each input observation without changing the schema fields themselves.

In [7]:
model.set(j2v.where("name") == "record", embed=True)

pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.5188801288604736,
│   │   │   │   -0.39736539125442505,
│   │   │   │   0.18413904309272766,
│   │   │   │   0.048634424805641174,
│   │   │   │   0.04227327182888985,
│   │   │   │   -0.11076778918504715,
│   │   │   │   0.37842732667922974,
│   │   │   │   0.008911083452403545,
│   │   │   │   -0.23850490152835846,
│   │   │   │   -0.18378086388111115,
│   │   │   │   0.17298369109630585,
│   │   │   │   0.3620986342430115,
│   │   │   │   0.31873995065689087,
│   │   │   │   -0.0457216277718544,
│   │   │   │   -0.11932995915412903,
│   │   │   │   0.09810446947813034
│   │   │   ],
│   │   │   [
│   │   │   │   -0.5188801288604736,
│   │   │   │   -0.39736539125442505,
│   │   │   │   0.18413904309272766,
│   │   │   │   0.048634424805641174,
│   │   │   │   0.04227327182888985,
│   │   │   │   -0.11076778918504715,
│   │   │   │   0.37842732667922974,
│   │   │   │   0.008911083452403545,
│   │   │   │   -0.23850490152835846,
│   │   │   │   -0.18378086388111115,
│   │   │   │   0.17298369109630585,
│   │   │   │   0.3620986342430115,
│   │   │   │   0.31873995065689087,
│   │   │   │   -0.0457216277718544,
│   │   │   │   -0.11932995915412903,
│   │   │   │   0.09810446947813034
│   │   │   ],
│   │   │   [
│   │   │   │   -0.5188801288604736,
│   │   │   │   -0.39736542105674744,
│   │   │   │   0.18413905799388885,
│   │   │   │   0.04863445460796356,
│   │   │   │   0.042273279279470444,
│   │   │   │   -0.11076777428388596,
│   │   │   │   0.37842726707458496,
│   │   │   │   0.008911140263080597,
│   │   │   │   -0.23850490152835846,
│   │   │   │   -0.18378092348575592,
│   │   │   │   0.17298372089862823,
│   │   │   │   0.3620986342430115,
│   │   │   │   0.31873998045921326,
│   │   │   │   -0.04572161287069321,
│   │   │   │   -0.11933000385761261,
│   │   │   │   0.09810442477464676
│   │   │   ]
│   │   ]
│   }
}

The plot is the quickest way to verify what was built: array nodes, tensorfield nodes, targets, embeddings, and parameter counts all appear in the same tree.

In [8]:
model.plot(detail=True)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                      │
│  JSON2Vec                                                             d_model  16                                    │
│  Model                                                                 params  18,153                                │
│  Schema map                                                            arrays  1                                     │
│                                                                        fields  3                                     │
│                                                                       targets  1                                     │
│                                                                        embeds  1                                     │
│                               